# Calibración de cámara

Flujo basado en el código de la práctica: detección y refinado de esquinas, generación de puntos del tablero y `cv2.calibrateCamera`. Se usa un tablero de 10 x 7  esquinas interiores y casillas de 20 mm.

In [1]:
import cv2
print("OpenCV should be 4.8.x Current version:", cv2.__version__)
from typing import List
import numpy as np
# import imageio --- IGNORE IF DEPRECATED---
import imageio.v3 as imageio
import copy
import os
import glob

OpenCV should be 4.8.x Current version: 4.8.0


In [2]:
def load_images(filenames: list) -> list:
    return [cv2.imread(filename) for filename in filenames]

# Build a list containing the paths of all images in the data folder
imgs_path = sorted(glob.glob("../data_calibracion/*"))
imgs = load_images(imgs_path)

In [6]:
print(imgs[0].shape)

(1008, 756, 3)


In [9]:
corners = [cv2.findChessboardCorners(img, (10,7)) for img in imgs]

In [11]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

# TODO To refine corner detections with cv2.cornerSubPix() you need to input grayscale images. Build a list containing grayscale images.
imgs_gray = [cv2.cvtColor(img,cv2.COLOR_BGR2GRAY) for img in imgs]

corners_refined = []
for img_gray, (found, c) in zip(imgs_gray, corners):
    if found:
        corners_refined.append(cv2.cornerSubPix(img_gray, c.copy(), (6, 6), (-1, -1), criteria))
    else:
        corners_refined.append(None)

In [15]:
imgs_copy = copy.deepcopy(imgs)
imgs_copy = [cv2.drawChessboardCorners(img,(10,7),corner_refined,True) for img,corner_refined in zip(imgs_copy,corners_refined) ]


In [16]:
def show_image(img):
    cv2.imshow("corner",img)
    cv2.waitKey()
    cv2.destroyAllWindows()
    
def write_image(name,img):
    cv2.imwrite(name,img)
    
for i,img in enumerate(imgs_copy):
    show_image(img)
    write_image(f"../Output/left/left_{i}.jpg",img)
   

In [17]:
def get_chessboard_points(chessboard_shape, dx, dy):
    cols, rows = chessboard_shape

    return np.array(
        [[dx * col, dy * row, 0]
         for row in range(rows)
         for col in range(cols)],
        dtype=np.float32
    )

In [18]:
points = get_chessboard_points((10, 7), 20, 20)

chessboard_points = [
    points.copy()
    for corner in corners
    if corner[0]
]


In [20]:
valid_corners = [c for c in corners_refined if c is not None]
valid_corners = np.array(valid_corners, dtype=np.float32)
valid_corners.shape

(18, 70, 1, 2)

In [22]:
image_size = imgs_gray[0].shape[::-1]
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
    chessboard_points,
    valid_corners,
    image_size,
    None,
    None
)
# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))

In [23]:
print("Intrinsics:\n", intrinsics)
print("Distortion coefficients:\n", dist_coeffs)
print("Root mean squared reprojection error:\n", rms)

Intrinsics:
 [[775.3770952    0.         367.08087401]
 [  0.         777.94353779 510.10299332]
 [  0.           0.           1.        ]]
Distortion coefficients:
 [[ 1.97599071e-01 -1.24057799e+00 -3.06253123e-04 -3.08980276e-03
   2.83269649e+00]]
Root mean squared reprojection error:
 0.43834689876953686
